<a href="https://colab.research.google.com/github/Mohamed-Shawky281/Hybrid-RAG-Research-assistant/blob/main/Hybrid_RAG_Research_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Hybrid RAG - Research Assistant Project

##Downloading libraries and dependencies...

In [1]:
!pip install -q langchain langchain-community langchain-chroma chromadb pypdf sentence-transformers rank_bm25
!pip install -q langchain-classic

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/rag_research_assistant"
os.makedirs(f"{PROJECT_DIR}/papers", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/processed", exist_ok=True)
print("Project folder ready at:", PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder ready at: /content/drive/MyDrive/rag_research_assistant


In [3]:
from pathlib import Path

pdf_paths = list(Path(f"{PROJECT_DIR}/papers").glob("*.pdf"))
print(f"{len(pdf_paths)} PDF(s) found in papers/:")
for p in pdf_paths:
    print(" -", p.name)

25 PDF(s) found in papers/:
 - CryptographyinPostQuantumComputingEra (1).pdf
 - DOC-20260822-WA0006_260915_154729 (1).pdf
 - DOC-20260827-WA0014_260915_154811 (1).pdf
 - DOC-20260827-WA0015_260915_155007 (1).pdf
 - DOC-20260827-WA0016_260915_155026 (1).pdf
 - CryptographyinPostQuantumComputingEra.pdf
 - DOC-20260827-WA0014_260915_154811.pdf
 - DOC-20260827-WA0016_260915_155026.pdf
 - DOC-20260822-WA0006_260915_154729.pdf
 - DOC-20260827-WA0015_260915_155007.pdf
 - CryptographyinPostQuantumComputingEra (2).pdf
 - DOC-20260822-WA0006_260915_154729 (2).pdf
 - DOC-20260827-WA0014_260915_154811 (2).pdf
 - DOC-20260827-WA0015_260915_155007 (2).pdf
 - DOC-20260827-WA0016_260915_155026 (2).pdf
 - CryptographyinPostQuantumComputingEra (3).pdf
 - DOC-20260822-WA0006_260915_154729 (3).pdf
 - DOC-20260827-WA0014_260915_154811 (3).pdf
 - DOC-20260827-WA0015_260915_155007 (3).pdf
 - DOC-20260827-WA0016_260915_155026 (3).pdf
 - CryptographyinPostQuantumComputingEra (4).pdf
 - DOC-20260822-WA0006_2609

Uploading The wanted reference documents

In [4]:
from google.colab import files

uploaded = files.upload()
for fname in uploaded.keys():
    dest = f"{PROJECT_DIR}/papers/{fname}"
    with open(dest, "wb") as f:
        f.write(uploaded[fname])
    print(f"Saved {fname} -> {dest}")

Saving CryptographyinPostQuantumComputingEra.pdf to CryptographyinPostQuantumComputingEra (5).pdf
Saving DOC-20260822-WA0006_260915_154729.pdf to DOC-20260822-WA0006_260915_154729 (5).pdf
Saving DOC-20260827-WA0014_260915_154811.pdf to DOC-20260827-WA0014_260915_154811 (5).pdf
Saving DOC-20260827-WA0015_260915_155007.pdf to DOC-20260827-WA0015_260915_155007 (5).pdf
Saving DOC-20260827-WA0016_260915_155026.pdf to DOC-20260827-WA0016_260915_155026 (5).pdf
Saved CryptographyinPostQuantumComputingEra (5).pdf -> /content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputingEra (5).pdf
Saved DOC-20260822-WA0006_260915_154729 (5).pdf -> /content/drive/MyDrive/rag_research_assistant/papers/DOC-20260822-WA0006_260915_154729 (5).pdf
Saved DOC-20260827-WA0014_260915_154811 (5).pdf -> /content/drive/MyDrive/rag_research_assistant/papers/DOC-20260827-WA0014_260915_154811 (5).pdf
Saved DOC-20260827-WA0015_260915_155007 (5).pdf -> /content/drive/MyDrive/rag_research_assistant

##Loading PDFs into LangChain Documents

In [5]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

pdf_paths = list(Path(f"{PROJECT_DIR}/papers").glob("*.pdf"))

all_docs = []
for path in pdf_paths:
    loader = PyPDFLoader(str(path))
    docs = loader.load()  # one Document per page, metadata already includes 'source' and 'page'
    all_docs.extend(docs)

print(f"Loaded {len(all_docs)} pages from {len(pdf_paths)} papers")
print(all_docs[0].page_content[:500])
print(all_docs[0].metadata)

/tmp/ipykernel_24414/455741346.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 810 pages from 30 papers
1 
CRYPTOGRAPHY IN POST-QUANTUM ERA 
 
 
 
 
 
Cryptography in Post Quantum Computing Era 
Neerav Sood 
Independent Researcher
{'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-01-24T13:31:21-05:00', 'author': 'TR', 'moddate': '2024-01-24T13:31:21-05:00', 'source': '/content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputingEra (1).pdf', 'total_pages': 96, 'page': 0, 'page_label': '1'}


In [6]:
#Take into consideration like truncated output , so strip it to avoid retreivial of gaps or empty spaces
import re

def clean_text(text: str) -> str:
    """
    Cleans up common PDF-extraction artifacts before chunking:
    - collapses multiple blank lines/newlines into one
    - collapses runs of spaces/tabs into a single space
    - strips leading/trailing whitespace per line
    - joins words that got hyphen-broken across a line (common in PDFs)
    """
    # Fix hyphenated words split across a line break, e.g. "crypto-\ngraphy" -> "cryptography"
    text = re.sub(r"-\n", "", text)

    # Collapse 3+ newlines (big gaps) down to a single paragraph break
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Collapse runs of spaces/tabs into one space
    text = re.sub(r"[ \t]+", " ", text)

    # Strip trailing whitespace on each line
    text = "\n".join(line.strip() for line in text.split("\n"))

    return text.strip()


for doc in all_docs:
    doc.page_content = clean_text(doc.page_content)

print("Cleaned text for all", len(all_docs), "pages")
print(all_docs[0].page_content[:500])  # sanity check -- compare to before cleaning

Cleaned text for all 810 pages
1
CRYPTOGRAPHY IN POST-QUANTUM ERA





Cryptography in Post Quantum Computing Era
Neerav Sood
Independent Researcher


##Chunking (including overlapping) & tuned

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,      # characters -- LangChain's default unit, Also to be tuned
    chunk_overlap=500,
    separators=["\n\n", "\n", ". ", " ", ""],  # tries paragraph breaks first, falls back to smaller units
)

split_docs = splitter.split_documents(all_docs)
print(f"Created {len(split_docs)} chunks from {len(all_docs)} pages")
print(split_docs[0].page_content[:300])
print(split_docs[0].metadata)

Created 2454 chunks from 810 pages
1
CRYPTOGRAPHY IN POST-QUANTUM ERA





Cryptography in Post Quantum Computing Era
Neerav Sood
Independent Researcher
{'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-01-24T13:31:21-05:00', 'author': 'TR', 'moddate': '2024-01-24T13:31:21-05:00', 'source': '/content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputingEra (1).pdf', 'total_pages': 96, 'page': 0, 'page_label': '1'}


##Building the Chroma vector store (Meaning Similarity)

In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embedding_model,
    persist_directory=f"{PROJECT_DIR}/chroma_db",
)

print(f"Chroma vector store built with {vectorstore._collection.count()} chunks")

/tmp/ipykernel_24414/2980357329.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Chroma vector store built with 10254 chunks


In [9]:
print(f"Chroma vector store built with {vectorstore._collection.count()} chunks")

Chroma vector store built with 10254 chunks


In [10]:
import pickle
import os

file_path = f"{PROJECT_DIR}/processed/split_docs.pkl"

# Check if the file exists. If not, and split_docs is available, save it.
# This assumes split_docs is defined in the current kernel state from previous cells.
if not os.path.exists(file_path):
    print(f"File '{file_path}' not found. Saving 'split_docs' to file first.")
    with open(file_path, "wb") as f:
        pickle.dump(split_docs, f)

# Now, attempt to load the file (which should now exist or existed already)
with open(file_path, "rb") as f:
    split_docs = pickle.load(f)

print(f"Loaded {len(split_docs)} chunks")

Loaded 920 chunks


##Build the Keyword similarity (BM25 retriever)

In [11]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(split_docs)
bm25_retriever.k = 5  # how many chunks BM25 returns per query

print("BM25 retriever built")

BM25 retriever built


In [12]:
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print("Vector retriever built")

Vector retriever built


##Merging of Both methods and tuning

In [13]:
from langchain_classic.retrievers.ensemble import EnsembleRetriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.65, 0.35], # The ratio of relavence used between vector and bm-25
)

print("Ensemble retriever built")

Ensemble retriever built


In [14]:
#Testing of retriver and accurate retrivial on a paper
query = "What wrong with Traditional cryptographic systems compared to quantum ones"

results = ensemble_retriever.invoke(query)

print(f"Returned {len(results)} chunks\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata.get('source', 'unknown')} | Page: {doc.metadata.get('page', '?')}")
    print(doc.page_content[:300])
    print()

Returned 3 chunks

--- Result 1 ---
Source: /content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputingEra.pdf | Page: 2
3 
CRYPTOGRAPHY IN POST-QUANTUM ERA 
 
 
Introduction 
1.1 Background and Significance 
In the realm of modern cryptography, the advent of quantum computing stands as a 
formidable challenge. Traditional cryptographic systems, which have long served as the bedrock 
of secure communication and data p

--- Result 2 ---
Source: /content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputingEra (1).pdf | Page: 19
20 
CRYPTOGRAPHY IN POST-QUANTUM ERA 
 
 
The robustness of lattice-based cryptographic systems against both classical and quantum 
attacks, coupled with their adaptability to a wide range of cryptographic applications, makes 
them a cornerstone in the quest for secure communication and data protect

--- Result 3 ---
Source: /content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputing

In [15]:
#Testing of retriver and accurate retrivial on a paper
query = "What problems are faced in multi-resource LLMs? "

results = ensemble_retriever.invoke(query)

print(f"Returned {len(results)} chunks\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata.get('source', 'unknown')} | Page: {doc.metadata.get('page', '?')}")
    print(doc.page_content[:300])
    print()

Returned 4 chunks

--- Result 1 ---
Source: /content/drive/MyDrive/rag_research_assistant/papers/DOC-20260827-WA0014_260915_154811 (1).pdf | Page: 3
necessitating extensive adjustments to other components.
1) Planning: First of all, to incorporate the cases which
does not require any sources of external knowledge, we define
several additional indicate tokens, corresponding to different
sources, including the NULL token which signifies that there

--- Result 2 ---
Source: /content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputingEra.pdf | Page: 14
complexity, making these problems intractable for both classical and quantum computers. 
The reason why lattice-based cryptographic methods are resistant to quantum computing 
attacks lies in the nature of these lattice problems. Unlike problems such as integer factorization 
or discrete logarithms,

--- Result 3 ---
Source: /content/drive/MyDrive/rag_research_assistant/papers/DOC-20260827-WA0016_260915_155026 (1).

In [24]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.4 MB/s eta 0:00:00


In [25]:
from google.colab import userdata
from groq import Groq

api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=api_key)

print("Groq client ready")

Groq client ready


In [27]:
import requests

resp = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {api_key}"}
)
models = resp.json()

for m in models["data"]:
    print(m["id"])

groq/compound-mini
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-arabic-saudi
qwen/qwen3.8-27b
openai/gpt-oss-120b
whisper-large-v3-turbo
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-86m
allam-2-7b
groq/compound
whisper-large-v3


In [26]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say hello in one short sentence."}]
)
print(response.choices[0].message.content)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}